In [2]:
#import libarries
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from enum import CONTINUOUS
import ipywidgets as widgets
from IPython.display import display


# Data download and undertsandoing

In [5]:
data=pd.read_csv('/content/sample_data/movies.csv')

In [6]:
data

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
62418,209157,We (2018),Drama
62419,209159,Window of the Soul (2001),Documentary
62420,209163,Bad Poems (2018),Comedy|Drama
62421,209169,A Girl Thing (2001),(no genres listed)


In [7]:
ratings =pd.read_csv('/content/sample_data/ratings.csv')

we have tow data base one for movies the other for ratings .


# Preprocesing the data

In [8]:
def clean_title(title):
  title=re.sub("[^a-zA-Z0-9 ]","",title)
  return title


In [ ]:
data['attractive title ']=data['title'].apply(clean_title)
data

,movieId,title,genres,clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
62418,209157,We (2018),Drama,We 2018
62419,209159,Window of the Soul (2001),Documentary,Window of the Soul 2001
62420,209163,Bad Poems (2018),Comedy|Drama,Bad Poems 2018
62421,209169,A Girl Thing (2001),(no genres listed),A Girl Thing 2001


In [ ]:
#build tfidf vector-transform the data
tfdf=TfidfVectorizer()
transform_data=tfdf.fit_transform(data['attractive title'])

In [14]:
#building an coollaborative   search function  for movies

def search(title):
  title=clean_title(title)
  qury_vec=tfdf.transform([title])
  similarity=cosine_similarity(qury_vec,transform_data).flatten()
  indecies=np.argpartition(similarity,-5)[-5:]
  results=data.iloc[indecies][: :-1]
  return results

In [15]:
sch=search('harry potter')
print(sch)

       movieId                                              title  \
4790      4896  Harry Potter and the Sorcerer's Stone (a.k.a. ...   
5704      5816     Harry Potter and the Chamber of Secrets (2002)   
11700    54001   Harry Potter and the Order of the Phoenix (2007)   
13512    69844      Harry Potter and the Half-Blood Prince (2009)   
10408    40815         Harry Potter and the Goblet of Fire (2005)   

                                       genres  \
4790               Adventure|Children|Fantasy   
5704                        Adventure|Fantasy   
11700            Adventure|Drama|Fantasy|IMAX   
13512  Adventure|Fantasy|Mystery|Romance|IMAX   
10408         Adventure|Fantasy|Thriller|IMAX   

                                                   clean  
4790   Harry Potter and the Sorcerers Stone aka Harry...  
5704        Harry Potter and the Chamber of Secrets 2002  
11700     Harry Potter and the Order of the Phoenix 2007  
13512         Harry Potter and the HalfBlood Prince 20

Building wedget for recomendation  smovies depending on its title  

In [ ]:
movie_input_name=widgets.Text(
    value='harry potter',
    description='Movie Title:',
    disabled=False,

)
recomendation_movie_list=widgets.Output()

def on_type(data):
  with recomendation_movie_list:
    recomendation_movie_list.clear_output()
    title=data["new"]
    if len(title)>5:
      results=search(title)
      if not results.empty:
                try:
                    display(results)
                except Exception as e:
                    print(f"Error finding recommendations: {e}")
      else:
                print("No movies found matching that title.")

movie_input_name.observe(on_type,names='value')
display(movie_input_name,recomendation_movie_list)

Text(value='Toy Story', description='Movie Title:')

Output()

finding users who like the same movies
and who like other movies

In [19]:
#build rocemndation function
def find_similar_movies(movie_id):
  # Find users who rated the given movie_id poorly
  similar_users = ratings[(ratings['movieId'] == movie_id) & (ratings['rating'] < 4)]['userId']

  if similar_users.empty:
    return pd.DataFrame(columns=['score', 'title', 'genres'])

  # Find all movies rated highly (e.g., > 4) by these similar users
  similar_user_high_ratings = ratings[
      ratings['userId'].isin(similar_users) & (ratings['rating'] > 4)
  ]

  # Calculate the percentage of similar users who highly rated each movie
  similar_users_movie_counts = similar_user_high_ratings.groupby('movieId')['userId'].nunique()

  if len(similar_users) == 0:
    similar_users_recs = pd.Series(dtype=float)
  else:
    similar_users_recs = similar_users_movie_counts / len(similar_users.unique())

  similar_users_recs = similar_users_recs[similar_users_recs > 0.1] # Retain the 0.1 threshold

  if similar_users_recs.empty:
    return pd.DataFrame(columns=['score', 'title', 'genres'])

  # Calculate the proportion of all users who rated each of these target movies highly
  target_movie_ids = similar_users_recs.index
  all_users_ratings_for_target_movies = ratings[
      (ratings['movieId'].isin(target_movie_ids)) & (ratings['rating'] > 4)
  ]
  total_unique_users = ratings['userId'].nunique()

  if total_unique_users == 0:
    all_users_rects = pd.Series(0.0, index=target_movie_ids)
  else:
    all_users_rects = all_users_ratings_for_target_movies.groupby('movieId')['userId'].nunique() / total_unique_users

  # Concatenate and calculate score
  react_percentage = pd.concat(
      [similar_users_recs, all_users_rects.reindex(similar_users_recs.index, fill_value=0)],
      axis=1
  )
  react_percentage.columns = ['similar', 'all']

  react_percentage['score'] = react_percentage['similar'] / react_percentage['all'].replace(0, np.nan)
  react_percentage = react_percentage.sort_values('score', ascending=False).dropna(subset=['score'])

  # Merge with movie data to get titles and genres
  final_recommendations = react_percentage.head(10).merge(
      data, left_index=True, right_on='movieId'
  )

  return final_recommendations[['score', 'title', 'genres']]

In [20]:
find_similar_movies(4929)


,score,title,genres
2994,22.265823,Scrooged (1988),Comedy|Fantasy|Romance
3509,20.897957,Pee-wee's Big Adventure (1985),Adventure|Comedy
1263,20.843557,Real Genius (1985),Comedy
3428,19.617465,Parenthood (1989),Comedy|Drama
1869,15.824372,Terms of Endearment (1983),Comedy|Drama
3898,15.269579,"Planes, Trains & Automobiles (1987)",Comedy
2300,13.613614,"Simple Plan, A (1998)",Crime|Drama|Thriller
930,13.562339,His Girl Friday (1940),Comedy|Romance
3005,13.494438,"Natural, The (1984)",Drama
4256,13.420176,Tootsie (1982),Comedy|Romance


Bulding interactive widget to search movies and it ratings with recomndations movies

In [21]:
#build function for searching a movies with its recomendations

movie_input_name=widgets.Text(
    value='Toy Story',
    description='Movie Title:',
    disabled=False,

)
recomendation_movie_list=widgets.Output()

def on_type(data):
  with recomendation_movie_list:
    recomendation_movie_list.clear_output()
    title=data["new"]
    if len(title)>5:
      results=search(title)
      if not results.empty:
                try:
                    # Get the ID of the first match
                    movie_id = results.iloc[0]['movieId']
                    # print(f"Movie ID: {movie_id}")

                    # 4. Display the recommendations
                    display(find_similar_movies(movie_id))
                except Exception as e:
                    print(f"Error finding recommendations: {e}")
      else:
                print("No movies found matching that title.")

movie_input_name.observe(on_type,names='value')
display(movie_input_name,recomendation_movie_list)

Text(value='Toy Story', description='Movie Title:')

Output()